# Solución optimizada CUDA Python — ASL

David Domínguez · 23712 / Jonathan Diaz · 23873 / Luis Padilla · 23663

**Ejecución nueva directa en este notebook:** `EXP-005` / `RUN-20261118T040625Z-cuda-notebook-01`.
Cada salida corresponde al código de la celda ejecutada. No se simula `fit` ni se
sustituye por leer un historial. La corrida es adicional a las cinco mediciones
del informe: sus promedios permanecen intactos. Al final se incluyen los registros
históricos completos identificados como tales.

Kernel: **PCUDA**. Preparación de archivos/imágenes en CPU.
Matemática del entrenamiento en GPU; validación y prueba en CPU.
Datos congelados:17,400 imágenes,RGB64×64,float32/255;train12180,val2610,test2610.
Se ejecutan15épocas completas por red,batch128,último20,sin EarlyStopping.

Las celdas operativas se extrajeron del AST del controlador medido y se separaron
por fase. Esta ejecución verifica su recorrido en notebook. Para repetir se debe
preparar otro EXP/RUN; este destino no se sobrescribe después de cerrarlo.


In [ ]:
from pathlib import Path
import ast, csv, hashlib, json, os, random, subprocess, sys, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/Proyecto 3 CUDA.pdf").is_file())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]
EXPERIMENT = ROOT / "resultados/paralelo/EXP-005"
CONFIG = ROOT / "configuraciones/EXP-005.json"
NOTEBOOK = ROOT / "LAB3-DATASCIENCE/Lab3_ASL_Deep_Learning.ipynb"
directory = EXPERIMENT / "RUN-20261118T040625Z-cuda-notebook-01"
repetition = 1
experiment = "EXP-005"
__file__ = str(ROOT / "scripts/benchmark_cuda.py")
assert not (directory / "salida.json").exists(), "Corrida cerrada: preparar otro EXP/RUN"
from IPython.display import display, Image, Markdown
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
print("Ejecución real:", experiment, directory.name)
print("Intérprete:", sys.executable)


## 1. Configuración, registro y funciones de medición

Las funciones siguientes registran tiempos y estado real; no entrenan al definirse.

In [ ]:
FIELDS = 'experimento_id,run_id,dispositivo,repeticion,fase,tipo_medicion,duracion_s,inicio_utc,fin_utc,epocas,pasos,muestras,batch_size,incluido_resumen,estado,configuracion_sha256,datos_sha256,evidencia,motivo'.split(',')

def utc():
    return datetime.now(timezone.utc).isoformat()

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def write_csv(path, rows, fields=None):
    with Path(path).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def original_namespace():
    """Compile only named definitions, preserving their original source bodies."""
    import numpy as np
    import pandas as pd
    from PIL import Image
    namespace = dict(np=np, pd=pd, Image=Image, random=random, SEED=42, IMG_SIZE=64)
    notebook = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
    names = {'build_sampled_file_list', 'load_images', 'build_cnn_a', 'build_cnn_b', 'build_fc'}
    for index, cell in enumerate(notebook['cells']):
        if cell['cell_type'] != 'code':
            continue
        tree = ast.parse(''.join(cell['source']))
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name in names:
                exec(compile(ast.Module(body=[node], type_ignores=[]), f'original/celda-{index}', 'exec'), namespace)
    namespace['_notebook'] = notebook
    return namespace

class Recorder:

    def __init__(self, directory, config, repetition):
        self.directory, self.config, self.repetition = (directory, config, repetition)
        self.rows = []
        self.epochs = []

    def state(self, phase, **more):
        write_json(self.directory / 'estado.json', dict(utc=utc(), fase=phase, **more))
        print(f'[{utc()}] {phase} {more}', flush=True)

    def measure(self, phase, function, **load):
        self.state(phase)
        start_utc, start = (utc(), time.perf_counter_ns())
        value = function()
        end, end_utc = (time.perf_counter_ns(), utc())
        self.add(phase, (end - start) / 1000000000.0, start_utc, end_utc, **load)
        return value

    def add(self, phase, seconds, start, end, **load):
        row = dict.fromkeys(FIELDS, '')
        row.update(experimento_id=self.config['experimento_id'], run_id=self.directory.name, dispositivo='cpu', repeticion=self.repetition, fase=phase, tipo_medicion='pared', duracion_s=seconds, inicio_utc=start, fin_utc=end, incluido_resumen='true', estado='completada', configuracion_sha256=sha(CONFIG), datos_sha256=self.config['datos']['manifest_splits_sha256'], evidencia=(self.directory / 'tiempos.csv').relative_to(ROOT).as_posix())
        row.update(load)
        self.rows.append(row)
        write_csv(self.directory / 'tiempos.csv', self.rows, FIELDS)
        print(f'RESULTADO {phase}: {seconds:.6f} s', flush=True)

In [ ]:
config = json.loads(CONFIG.read_text(encoding="utf8"))
print(json.dumps(config, indent=2, ensure_ascii=False))
from registro_notebook import iniciar_monitor
monitor_stop, monitor_thread = iniciar_monitor(directory)


## 2. Datos y funciones originales utilizadas

Código original conservado: carga de imágenes y arquitectura. Las tablas muestran todos los conteos de clases y particiones.

In [ ]:
def build_cnn_b():

    def bn():
        return layers.BatchNormalization(momentum=0.9)
    model = keras.Sequential([layers.Input((IMG_SIZE, IMG_SIZE, 3)), layers.Conv2D(32, 3, padding='same', activation='relu'), bn(), layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25), layers.Conv2D(64, 3, padding='same', activation='relu'), bn(), layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25), layers.Conv2D(128, 3, padding='same', activation='relu'), bn(), layers.MaxPooling2D(), layers.Dropout(0.3), layers.Flatten(), layers.Dense(256, activation='relu'), bn(), layers.Dropout(0.4), layers.Dense(NUM_CLASSES, activation='softmax')], name='CNN_B_regularizada')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
datos_manifest = pd.read_csv(ROOT / "resultados/secuencial/EXP-002/manifest_splits.csv", keep_default_na=False)
display(pd.crosstab(datos_manifest.clase, datos_manifest.split, margins=True))
display(datos_manifest.head(12))
print("17,400 rutas/identidades completas en el manifiesto congelado; no se vuelve a muestrear.")


## 3. CUDA Python: configuración y código completo de los módulos

Cada celda `%%modulo` ejecuta la fuente Python mostrada en un módulo de memoria. No escribe ni altera los archivos del proyecto. Los kernels y la red usados abajo son estos mismos módulos.

In [ ]:
import types
from IPython.core.magic import register_cell_magic
pkg = types.ModuleType("asl_cuda")
pkg.__path__ = [str(ROOT / "src/asl_cuda")]
sys.modules["asl_cuda"] = pkg
@register_cell_magic
def modulo(nombre, codigo):
    module = types.ModuleType(nombre)
    module.__file__ = str(ROOT / "src" / (nombre.replace(".", "/") + ".py"))
    module.__package__ = nombre.rsplit(".", 1)[0]
    sys.modules[nombre] = module
    setattr(sys.modules["asl_cuda"], nombre.rsplit(".", 1)[-1], module)
    exec(compile(codigo, module.__file__, "exec"), module.__dict__)
    print("Módulo ejecutado:", nombre, "SHA-256:", hashlib.sha256(codigo.encode()).hexdigest())


In [ ]:
%%modulo asl_cuda.runtime
"""Local toolkit discovery. Never installs packages or silently falls back to CPU."""
import os
from pathlib import Path

_DLL_HANDLES = []


def configure():
    if os.environ.get("NUMBA_ENABLE_CUDASIM", "0") != "0":
        raise RuntimeError("CUDA simulator is not a GPU benchmark.")
    if os.name == "nt":
        root = Path(os.environ.get("CUDA_HOME", "C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v13.1"))
        os.environ.setdefault("CUDA_HOME", str(root))
        for folder in (root / "bin/x64", root / "bin", root / "nvvm/bin/x64", root / "nvvm/bin"):
            if folder.is_dir():
                _DLL_HANDLES.append(os.add_dll_directory(str(folder)))


configure()
# numba-cuda 0.30.4 still registers NumPy's retired row_stack alias at import.
# Preserve the installed NumPy 2.5.5: the alias is exactly vstack, not new math.
# Process-local compatibility only; no site-packages files are changed.
import numpy as np
if not hasattr(np, "row_stack"):
    np.row_stack = np.vstack
from numba import cuda  # noqa: E402


def device_info():
    if not cuda.is_available():
        raise RuntimeError("A real CUDA device is required.")
    d = cuda.get_current_device()
    free, total = cuda.current_context().get_memory_info()
    return dict(name=d.name.decode() if isinstance(d.name, bytes) else d.name,
                compute_capability=list(d.compute_capability), free_bytes=int(free), total_bytes=int(total),
                max_threads_per_block=int(d.MAX_THREADS_PER_BLOCK),
                max_shared_memory_per_block=int(d.MAX_SHARED_MEMORY_PER_BLOCK),
                simulator=False)


In [ ]:
%%modulo asl_cuda.kernels
"""K01-K19, DIS-CUDA-001: FP32 NHWC, gather backward, explicit reductions.

Flat buffers have explicit active dimensions; capacity never defines batch size.
No floating point atomics or cross-block barriers. Partial gradients are unique.
"""
import math
from numba import float32
from .runtime import cuda


@cuda.jit(device=True, inline=True)
def _left(a, mode, m, k, H, W, Ci, Co, I, O):
    if mode == 0:  # virtual im2col
        n = m // (H * W)
        h = (m // W) % H + k // (3 * Ci) - 1
        w = m % W + (k // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + k % Ci]
        return float32(0)
    if mode == 1:  # gather dX, spatially reversed offsets
        n = m // (H * W)
        h = (m // W) % H - k // (3 * Co) + 1
        w = m % W - (k // Co) % 3 + 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Co + k % Co]
        return float32(0)
    if mode == 2:  # im2col transpose without materializing it
        n = k // (H * W)
        h = (k // W) % H + m // (3 * Ci) - 1
        w = k % W + (m // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + m % Ci]
        return float32(0)
    if mode == 3:
        return a[m * I + k]
    if mode == 4:
        return a[m * O + k]
    return a[k * I + m]


@cuda.jit(device=True, inline=True)
def _right(v, mode, k, n, Ci, Co, I, O):
    if mode == 0 or mode == 2:
        return v[k * Co + n]
    if mode == 1:
        return v[((k // Co) * Ci + n) * Co + k % Co]
    if mode == 3 or mode == 5:
        return v[k * O + n]
    return v[n * O + k]


def _make_gemm(mode):
    @cuda.jit(cache=True, fastmath=False)
    def kernel(a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0):
        sa = cuda.shared.array((16, 16), float32)
        sb = cuda.shared.array((16, 16), float32)
        tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
        m, n = m0 + cuda.blockIdx.y * 16 + ty, cuda.blockIdx.x * 16 + tx
        begin, end = 0, K
        if mode == 2 or mode == 5:
            begin = cuda.blockIdx.z * 2048
            end = min(K, begin + 2048)
        acc = float32(0)
        for k0 in range(begin, end, 16):
            sa[ty, tx] = float32(0)
            sb[ty, tx] = float32(0)
            if m < M and k0 + tx < end:
                sa[ty, tx] = _left(a, mode, m, k0 + tx, H, W, Ci, Co, I, O)
            if n < N and k0 + ty < end:
                sb[ty, tx] = _right(v, mode, k0 + ty, n, Ci, Co, I, O)
            cuda.syncthreads()
            for kk in range(16):
                acc += sa[ty, kk] * sb[kk, tx]
            cuda.syncthreads()
        if m < M and n < N:
            index = m * N + n
            if mode == 2 or mode == 5:
                index += cuda.blockIdx.z * M * N
            if mode == 0 or mode == 3:
                acc += bias[n]
                if relu and acc < 0:
                    acc = float32(0)
            out[index] = acc
    return kernel


GEMM = tuple(_make_gemm(i) for i in range(6))


@cuda.jit(cache=True)
def merge_partials(parts, out, E, P):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = float32(0)
        for p in range(P):
            value += parts[p * E + i]
        out[i] = value


@cuda.jit(cache=True)
def copy_flat(x, y, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        y[i] = x[i]


@cuda.jit(cache=True)
def relu_backward(grad, activation, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if activation[i] <= 0:
            grad[i] = float32(0)


@cuda.jit(cache=True)
def dropout(x, mask, out, factor, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        out[i] = x[i] * float32(mask[i]) * factor


@cuda.jit(cache=True)
def pool_forward(x, y, arg, b, H, W, C):
    E = b * (H // 2) * (W // 2) * C
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        w = (i // C) % (W // 2)
        h = (i // (C * (W // 2))) % (H // 2)
        n = i // (C * (W // 2) * (H // 2))
        base = ((n * H + 2 * h) * W + 2 * w) * C + c
        value, pos = x[base], 0
        for p in range(1, 4):
            candidate = x[base + (p // 2) * W * C + (p % 2) * C]
            if candidate > value:
                value, pos = candidate, p
        y[i], arg[i] = value, pos


@cuda.jit(cache=True)
def pool_backward(dy, arg, dx, b, H, W, C):
    for i in range(cuda.grid(1), b * H * W * C, cuda.gridsize(1)):
        c, w = i % C, (i // C) % W
        h, n = (i // (C * W)) % H, i // (C * W * H)
        out_i = ((n * (H // 2) + h // 2) * (W // 2) + w // 2) * C + c
        dx[i] = float32(0)
        if arg[out_i] == 2 * (h % 2) + w % 2:
            dx[i] = dy[out_i]


@cuda.jit(cache=True, fastmath=False)
def reduce_channels(x, dy, mean, inv, parts1, parts2, L, C, mode):
    # mode0 sum; mode1 centered variance; mode2 BN backward pair.
    s1 = cuda.shared.array((8, 32), float32)
    s2 = cuda.shared.array((8, 32), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c = cuda.blockIdx.y * 32 + tx
    v1, v2 = float32(0), float32(0)
    if c < C:
        for j in range(256):
            l = cuda.blockIdx.x * 2048 + ty + 8 * j
            if l < L:
                value = x[l * C + c]
                if mode == 0:
                    v1 += value
                elif mode == 1:
                    centered = value - mean[c]
                    v1 += centered * centered
                else:
                    g = dy[l * C + c]
                    v1 += g
                    v2 += g * ((value - mean[c]) * inv[c])
    s1[ty, tx], s2[ty, tx] = v1, v2
    cuda.syncthreads()
    stride = 4
    while stride > 0:
        if ty < stride:
            s1[ty, tx] += s1[ty + stride, tx]
            s2[ty, tx] += s2[ty + stride, tx]
        cuda.syncthreads()
        stride //= 2
    if ty == 0 and c < C:
        parts1[cuda.blockIdx.x * C + c] = s1[0, tx]
        parts2[cuda.blockIdx.x * C + c] = s2[0, tx]


@cuda.jit(cache=True, fastmath=False)
def finish_channels(p1, p2, out1, out2, movmean, movvar, mean, inv, P, C, L, mode):
    # mode0 mean; mode1 variance+moving state; mode2 dbeta/dgamma; mode3 bias.
    s1 = cuda.shared.array(256, float32)
    s2 = cuda.shared.array(256, float32)
    t, c = cuda.threadIdx.x, cuda.blockIdx.x
    s1[t], s2[t] = float32(0), float32(0)
    if t < P:
        s1[t], s2[t] = p1[t * C + c], p2[t * C + c]
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s1[t] += s1[t + stride]
            s2[t] += s2[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        if mode == 0:
            out1[c] = s1[0] / float32(L)
        elif mode == 1:
            var = s1[0] / float32(L)
            inv[c] = float32(1) / math.sqrt(var + float32(.001))
            movmean[c] = float32(.9) * movmean[c] + float32(.1) * mean[c]
            movvar[c] = float32(.9) * movvar[c] + float32(.1) * var
        elif mode == 2:
            out1[c], out2[c] = s1[0], s2[0]
        else:
            out1[c] = s1[0]


@cuda.jit(cache=True, fastmath=False)
def bn_forward(x, y, gamma, beta, mean, inv, E, C):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        y[i] = gamma[c] * ((x[i] - mean[c]) * inv[c]) + beta[c]


@cuda.jit(cache=True, fastmath=False)
def bn_backward(x, dy, dx, gamma, mean, inv, dbeta, dgamma, L, C):
    for i in range(cuda.grid(1), L * C, cuda.gridsize(1)):
        c = i % C
        xhat = (x[i] - mean[c]) * inv[c]
        dx[i] = (gamma[c] * inv[c] / float32(L)) * (
            float32(L) * dy[i] - dbeta[c] - xhat * dgamma[c])


@cuda.jit(cache=True, fastmath=False)
def softmax_loss(logits, labels, probs, dlogits, loss, correct, b):
    s = cuda.shared.array(32, float32)
    t, n = cuda.threadIdx.x, cuda.blockIdx.x
    z = float32(-math.inf)
    if t < 29:
        z = logits[n * 29 + t]
    s[t] = z
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] = max(s[t], s[t + stride])
        cuda.syncthreads()
        stride //= 2
    maximum = s[0]
    cuda.syncthreads()
    value = float32(0)
    if t < 29:
        value = math.exp(z - maximum)
    s[t] = value
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
        cuda.syncthreads()
        stride //= 2
    denominator = s[0]
    label = labels[n]
    if t < 29:
        probability = value / denominator
        probs[n * 29 + t] = probability
        dlogits[n * 29 + t] = (probability - float32(t == label)) / float32(b)
    if t == 0:
        loss[n] = (maximum - logits[n * 29 + label]) + math.log(denominator)
        winner = 0
        for c in range(1, 29):
            if logits[n * 29 + c] > logits[n * 29 + winner]:
                winner = c
        correct[n] = winner == label


@cuda.jit(cache=True, fastmath=False)
def adam(theta, grad, moment, velocity, alpha, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        g = grad[i]
        if not math.isfinite(g):
            cuda.atomic.max(error, 0, 1)
        if error[0] == 0:
            m = moment[i] + (g - moment[i]) * float32(.1)
            v = velocity[i] + (g * g - velocity[i]) * float32(.001)
            value = theta[i] - (m * alpha) / (math.sqrt(v) + float32(1e-7))
            if not math.isfinite(value):
                cuda.atomic.max(error, 0, 2)
            theta[i], moment[i], velocity[i] = value, m, v


@cuda.jit(cache=True)
def finite_check(x, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if not math.isfinite(x[i]):
            cuda.atomic.max(error, 0, 3)


@cuda.jit(cache=True)
def metrics(loss, correct, b, total):
    s = cuda.shared.array(256, float32)
    q = cuda.shared.array(256, float32)
    t = cuda.threadIdx.x
    s[t], q[t] = float32(0), float32(0)
    if t < b:
        s[t], q[t] = loss[t], float32(correct[t])
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
            q[t] += q[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        total[0] += s[0]
        total[1] += q[0]
        total[2] += float32(b)


@cuda.jit(cache=True)
def advance_step(counter):
    if cuda.grid(1) == 0:
        counter[0] += 1


@cuda.jit(cache=True)
def im2col(x, cols, R, H, W, Ci):
    K = 9 * Ci
    for i in range(cuda.grid(1), R * K, cuda.gridsize(1)):
        r, kk = i // K, i % K
        n = r // (H * W)
        h = (r // W) % H + kk // (3 * Ci) - 1
        w = r % W + (kk // Ci) % 3 - 1
        value = float32(0)
        if 0 <= h < H and 0 <= w < W:
            value = x[((n * H + h) * W + w) * Ci + kk % Ci]
        cols[i] = value


@cuda.jit(cache=True)
def col2im_gather(cols, dx, R, H, W, Ci):
    for i in range(cuda.grid(1), R * Ci, cuda.gridsize(1)):
        c = i % Ci
        w, h = (i // Ci) % W, (i // (Ci * W)) % H
        n = i // (Ci * W * H)
        value = float32(0)
        for kh in range(3):
            for kw in range(3):
                ho, wo = h - kh + 1, w - kw + 1
                if 0 <= ho < H and 0 <= wo < W:
                    row = (n * H + ho) * W + wo
                    value += cols[row * 9 * Ci + (kh * 3 + kw) * Ci + c]
        dx[i] = value


@cuda.jit(cache=True)
def bias_activation(y, bias, E, C, relu):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = y[i] + bias[i % C]
        if relu and value < 0:
            value = float32(0)
        y[i] = value


In [ ]:
%%modulo asl_cuda.blas
"""FP32 cuBLAS SGEMM on Numba-owned device memory and the SAME CUDA stream.

The installed NVIDIA CUDA toolkit supplies the DLL. No tensor cores, no TF32,
no NumPy/CPU multiplication. Geometry inside cuBLAS belongs to NVIDIA's backend.
"""
import ctypes as ct
import os
from pathlib import Path


class Blas:
    def __init__(self, stream):
        root = Path(os.environ["CUDA_HOME"])
        self.path = root / "bin/x64/cublas64_13.dll"
        self.dll = ct.WinDLL(str(self.path))
        self.handle = ct.c_void_p()
        self._signature("cublasCreate_v2", [ct.POINTER(ct.c_void_p)])
        self._signature("cublasSetStream_v2", [ct.c_void_p, ct.c_void_p])
        self._signature("cublasSetMathMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasSetAtomicsMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasGetVersion_v2", [ct.c_void_p, ct.POINTER(ct.c_int)])
        self._signature("cublasDestroy_v2", [ct.c_void_p])
        self._signature("cublasSgemm_v2", [ct.c_void_p, ct.c_int, ct.c_int,
            ct.c_int, ct.c_int, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int,
            ct.c_void_p, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int])
        self._call("cublasCreate_v2", ct.byref(self.handle))
        raw = stream.handle
        self._call("cublasSetStream_v2", self.handle, ct.c_void_p(raw.value if hasattr(raw, "value") else int(raw)))
        self._call("cublasSetMathMode", self.handle, 2)  # CUBLAS_PEDANTIC_MATH
        self._call("cublasSetAtomicsMode", self.handle, 0)  # CUBLAS_ATOMICS_NOT_ALLOWED
        version = ct.c_int()
        self._call("cublasGetVersion_v2", self.handle, ct.byref(version))
        self.version = version.value

    def _signature(self, name, args):
        fn = getattr(self.dll, name)
        fn.argtypes, fn.restype = args, ct.c_int

    def _call(self, name, *args):
        status = getattr(self.dll, name)(*args)
        if status:
            raise RuntimeError(f"{name}: cuBLAS status {status}")

    @staticmethod
    def ptr(array):
        return ct.c_void_p(array.device_ctypes_pointer.value)

    def gemm(self, a, b, out, M, N, K, transpose_a=False, transpose_b=False):
        """C_row=op(A)_row @ op(B)_row, implemented as C_col=B_col @ A_col."""
        alpha, beta = ct.c_float(1.), ct.c_float(0.)
        self._call("cublasSgemm_v2", self.handle, int(transpose_b), int(transpose_a),
                   N, M, K, ct.byref(alpha), self.ptr(b), K if transpose_b else N,
                   self.ptr(a), M if transpose_a else K, ct.byref(beta), self.ptr(out), N)

    def close(self):
        if self.handle:
            self._call("cublasDestroy_v2", self.handle)
            self.handle = ct.c_void_p()


In [ ]:
%%modulo asl_cuda.model
"""Complete CNN-B training on explicitly launched CUDA Python kernels.

TensorFlow is used only as a CPU oracle and for CPU validation/checkpoint I/O.
The original network and its initial state are imported without editing them.
"""
from contextlib import contextmanager
from pathlib import Path
import hashlib
import json
import time
import numpy as np
from .runtime import cuda, device_info
from . import kernels as k

ROOT = Path(__file__).resolve().parents[2]
INITIAL = ROOT / "data/EXP-002/RUN-20261117T171942Z-cpu-01/CNN-B.estado-inicial.npz"
INITIAL_SHA = "10558d723213752b759626c3b196665f2547a0b65cbce296e47b3a4f992305fd"


def cpu_reference():
    """Build original definitions, confirm CPU placement and restore ALL 80 arrays."""
    import os
    for name, value in {"TF_NUM_INTRAOP_THREADS":"6", "TF_NUM_INTEROP_THREADS":"1",
                        "OMP_NUM_THREADS":"6", "OPENBLAS_NUM_THREADS":"6",
                        "MKL_NUM_THREADS":"6", "TF_DETERMINISTIC_OPS":"1",
                        "TF_ENABLE_ONEDNN_OPTS":"1"}.items():
        os.environ[name] = value
    import tensorflow as tf
    tf.config.set_visible_devices([], "GPU")
    tf.config.threading.set_intra_op_parallelism_threads(6)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.utils.set_random_seed(43)
    tf.keras.mixed_precision.set_global_policy("float32")
    import sys
    sys.path.insert(0, str(ROOT / "scripts"))
    from benchmark_cpu import original_namespace
    ns = original_namespace()
    ns.update(keras=tf.keras, layers=tf.keras.layers, NUM_CLASSES=29)
    model = ns["build_cnn_b"]()
    model.jit_compile = False
    model.optimizer.build(model.trainable_variables)
    if hashlib.sha256(INITIAL.read_bytes()).hexdigest() != INITIAL_SHA:
        raise ValueError("Initial checkpoint identity changed")
    with np.load(INITIAL, allow_pickle=False) as archive:
        variables = list(model.variables) + list(model.optimizer.variables)
        if len(variables) != len(archive.files):
            raise ValueError("Initial state variable count differs")
        for index, variable in enumerate(variables):
            value = archive[f"arr_{index}"]
            if tuple(variable.shape) != value.shape or str(variable.dtype) != str(value.dtype):
                raise ValueError(f"Incompatible state variable {index}")
            variable.assign(value)
    if int(model.optimizer.iterations.numpy()) != 0:
        raise ValueError("Expected optimizer iteration zero")
    if any(np.any(v.numpy()) for v in [*model.optimizer._momentums, *model.optimizer._velocities]):
        raise ValueError("Nonzero initial Adam requires explicit state mapping")
    return tf, model


def grid(E):
    return max(1, min(4096, (int(E) + 255) // 256))


def compiled_resources():
    """Numba-CUDA 0.30 API expects signature objects, not .signatures tuples."""
    result={}
    for name,kern in vars(k).items():
        if hasattr(kern,"get_regs_per_thread") and kern.signatures:
            regs=kern.get_regs_per_thread()
            shared=kern.get_shared_mem_per_block()
            local=kern.get_local_mem_per_thread()
            result[name]=[dict(firma=str(sig),registros=value,shared_bytes=shared[sig],local_bytes=local[sig])
                          for sig,value in regs.items()]
    return result


class Network:
    def __init__(self, reference, backend="explicit"):
        self.info = device_info()
        if self.info["free_bytes"] < 1536 * 2**20:
            raise MemoryError("Insufficient free VRAM for the frozen profile")
        self.stream = cuda.stream()
        if backend not in ("explicit", "cublas"):
            raise ValueError("Unknown backend")
        self.backend = backend
        self.allocations, self.params, self.ops, self.transfers = [], [], [], []
        self.reference = reference
        self.input = self.alloc(128 * 64 * 64 * 3)
        self.host_x = cuda.pinned_array(128 * 64 * 64 * 3, dtype=np.float32)
        self.labels = self.alloc(128, np.int64)
        self.host_y = cuda.pinned_array(128, dtype=np.int64)
        shape, previous = (64, 64, 3), self.input
        for layer in reference.layers:
            name = type(layer).__name__
            item = dict(input=previous, shape=shape, layer=layer)
            if name == "Conv2D":
                H, W, Ci = shape
                Co = layer.filters
                item.update(kind="conv", H=H, W=W, Ci=Ci, Co=Co,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias), relu=True)
                shape = (H, W, Co)
            elif name == "Dense":
                I, O = int(layer.kernel.shape[0]), int(layer.kernel.shape[1])
                item.update(kind="dense", I=I, O=O, relu=O != 29,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias))
                shape = (O,)
            elif name == "BatchNormalization":
                C = shape[-1]
                item.update(kind="bn", C=C, gamma=self.param(layer.gamma), beta=self.param(layer.beta),
                            moving_mean=self.alloc(C), moving_var=self.alloc(C),
                            initial_mean=layer.moving_mean.numpy().copy(), initial_var=layer.moving_variance.numpy().copy(),
                            mean=self.alloc(C), inv=self.alloc(C))
            elif name == "MaxPooling2D":
                H, W, C = shape
                item.update(kind="pool", H=H, W=W, C=C)
                shape = (H // 2, W // 2, C)
                item["arg"] = self.alloc(128 * int(np.prod(shape)), np.uint8)
            elif name == "Dropout":
                E = 128 * int(np.prod(shape))
                item.update(kind="drop", p=layer.rate, mask=self.alloc(E, np.uint8),
                            host_mask=cuda.pinned_array(E, dtype=np.uint8))
            elif name == "Flatten":
                shape = (int(np.prod(shape)),)
                continue
            elif name == "InputLayer":
                continue
            else:
                raise TypeError(f"Unsupported original layer {name}")
            item["sample_elements"] = int(np.prod(shape))
            item["output"] = self.alloc(128 * item["sample_elements"])
            self.ops.append(item)
            previous = item["output"]
        self.probs = self.alloc(128 * 29)
        self.loss, self.correct = self.alloc(128), self.alloc(128, np.uint8)
        self.g1, self.g2 = self.alloc(128 * 64 * 64 * 32), self.alloc(128 * 64 * 64 * 32)
        self.parts = self.alloc(16 * 2**20 // 4)
        self.p1, self.p2 = self.alloc(65536), self.alloc(65536)
        self.dummy = self.alloc(256)
        self.total = self.alloc(3)
        self.error = self.alloc(1, np.int32)
        self.counter = self.alloc(1, np.int64)
        self.host_error = cuda.pinned_array(1, dtype=np.int32)
        self.host_total = cuda.pinned_array(3, dtype=np.float32)
        if backend == "cublas":
            from .blas import Blas
            self.blas = Blas(self.stream)
            self.columns = self.alloc(128 * 64 * 64 * (9 * 32))
        self.bytes_allocated = sum(a.nbytes for a in self.allocations)
        if self.bytes_allocated > (1408 if backend == "cublas" else 768) * 2**20:
            raise MemoryError(f"Arena exceeds design: {self.bytes_allocated}")
        if sum(p["value"].size for p in self.params) != 2245245:
            raise ValueError("Architecture parameter count changed")
        self.events = []
        self.restore()

    def alloc(self, count, dtype=np.float32):
        array = cuda.device_array(int(count), dtype=dtype, stream=self.stream)
        self.allocations.append(array)
        return array

    def param(self, variable):
        values = variable.numpy().copy()
        p = dict(initial=values, value=self.alloc(values.size), grad=self.alloc(values.size),
                 m=self.alloc(values.size), v=self.alloc(values.size), variable=variable)
        self.params.append(p)
        return p

    def restore(self):
        for p in self.params:
            p["value"].copy_to_device(p["initial"].ravel(), stream=self.stream)
            for state in ("m", "v"):
                p[state].copy_to_device(np.zeros(p["initial"].size, np.float32), stream=self.stream)
        for op in self.ops:
            if op["kind"] == "bn":
                op["moving_mean"].copy_to_device(op["initial_mean"], stream=self.stream)
                op["moving_var"].copy_to_device(op["initial_var"], stream=self.stream)
        for a in (self.error, self.counter, self.total):
            a.copy_to_device(np.zeros(a.size, dtype=a.dtype), stream=self.stream)
        self.stream.synchronize()
        self.steps = 0
        self.events = []

    @contextmanager
    def timed(self, phase):
        start, end = cuda.event(timing=True), cuda.event(timing=True)
        start.record(self.stream)
        yield
        end.record(self.stream)
        self.events.append((phase, start, end))

    def drain_events(self):
        self.stream.synchronize()
        result = {}
        for phase, start, end in self.events:
            result[phase] = result.get(phase, 0.) + cuda.event_elapsed_time(start, end) / 1000.
        self.events.clear()
        return result

    def gemm(self, mode, a, v, bias, out, M, N, K, H=1, W=1, Ci=1, Co=1, I=1, O=1, relu=False):
        if self.backend == "cublas":
            if mode in (0, 2):
                R = M if mode == 0 else K
                width = 9 * Ci
                k.im2col[grid(R * width), 256, self.stream](a, self.columns, R, H, W, Ci)
                self.blas.gemm(self.columns, v, out, M, N, K, transpose_a=mode == 2)
            elif mode == 1:
                self.blas.gemm(a, v, self.columns, M, 9 * Ci, Co, transpose_b=True)
                k.col2im_gather[grid(M * Ci), 256, self.stream](self.columns, out, M, H, W, Ci)
            else:
                self.blas.gemm(a, v, out, M, N, K, transpose_a=mode == 5, transpose_b=mode == 4)
            if mode in (0, 3):
                k.bias_activation[grid(M * N), 256, self.stream](out, bias, M * N, N, relu)
            return
        P = (K + 2047) // 2048 if mode in (2, 5) else 1
        if mode in (2, 5) and P * M * N > self.parts.size:
            raise MemoryError("Partial gradient workspace exhausted")
        for m0 in range(0, M, 16384):
            geom = ((N + 15) // 16, (min(16384, M - m0) + 15) // 16, P)
            k.GEMM[mode][geom, (16, 16), self.stream](
                a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0)

    def reduce(self, x, dy, op, L, C, mode, out1, out2):
        P = (L + 2047) // 2048
        if P > 256 or P * C > self.p1.size:
            raise ValueError("Reduction shape exceeds frozen profile")
        mean, inv = op.get("mean", self.dummy), op.get("inv", self.dummy)
        k.reduce_channels[(P, (C + 31) // 32), (32, 8), self.stream](
            x, dy, mean, inv, self.p1, self.p2, L, C, 2 if mode == 2 else (1 if mode == 1 else 0))
        k.finish_channels[C, 256, self.stream](self.p1, self.p2, out1, out2,
            op.get("moving_mean", self.dummy), op.get("moving_var", self.dummy),
            mean, inv, P, C, L, mode)

    def upload(self, x, labels, epoch, batch_index):
        b = len(x)
        np.copyto(self.host_x[:x.size], x.ravel())
        np.copyto(self.host_y[:b], labels)
        drops = [op for op in self.ops if op["kind"] == "drop"]
        for d, op in enumerate(drops):
            E = b * op["sample_elements"]
            rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([43, epoch, batch_index, d + 1])))
            op["host_mask"][:E] = rng.random(E, dtype=np.float32) >= np.float32(op["p"])
        with self.timed("h2d_lotes"):
            self.input[:x.size].copy_to_device(self.host_x[:x.size], stream=self.stream)
            self.labels[:b].copy_to_device(self.host_y[:b], stream=self.stream)
            for op in drops:
                E = b * op["sample_elements"]
                op["mask"][:E].copy_to_device(op["host_mask"][:E], stream=self.stream)
        self.transfers.append(dict(batch=b, imagenes_bytes=x.nbytes, etiquetas_bytes=labels.nbytes,
                                   mascaras_bytes=sum(b * op["sample_elements"] for op in drops)))

    def forward(self, b):
        for op in self.ops:
            kind, x, y, E = op["kind"], op["input"], op["output"], b * op["sample_elements"]
            if kind == "conv":
                H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                self.gemm(0, x, op["weights"]["value"], op["bias"]["value"], y,
                          b * H * W, Co, 9 * Ci, H, W, Ci, Co, relu=True)
            elif kind == "dense":
                self.gemm(3, x, op["weights"]["value"], op["bias"]["value"], y,
                          b, op["O"], op["I"], I=op["I"], O=op["O"], relu=op["relu"])
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, self.dummy, op, L, C, 0, op["mean"], self.dummy)
                self.reduce(x, self.dummy, op, L, C, 1, self.dummy, self.dummy)
                k.bn_forward[grid(E), 256, self.stream](x, y, op["gamma"]["value"],
                    op["beta"]["value"], op["mean"], op["inv"], E, C)
            elif kind == "pool":
                k.pool_forward[grid(E), 256, self.stream](x, y, op["arg"], b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](x, op["mask"], y, np.float32(1 / (1 - op["p"])), E)
            k.finite_check[grid(E), 256, self.stream](y, E, self.error)
        k.softmax_loss[b, 32, self.stream](self.ops[-1]["output"], self.labels,
            self.probs, self.g1, self.loss, self.correct, b)
        k.finite_check[grid(b), 256, self.stream](self.loss, b, self.error)

    def backward(self, b, audit_hook=None):
        current, other = self.g1, self.g2
        for index in range(len(self.ops) - 1, -1, -1):
            op = self.ops[index]
            kind, x = op["kind"], op["input"]
            E = b * op["sample_elements"]
            if kind in ("conv", "dense"):
                if op["relu"]:
                    k.relu_backward[grid(E), 256, self.stream](current, op["output"], E)
                if kind == "conv":
                    H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                    L = b * H * W
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(2, x, current, self.dummy, target, 9 * Ci, Co, L, H, W, Ci, Co)
                    if self.backend == "explicit":
                        k.merge_partials[grid(9 * Ci * Co), 256, self.stream](self.parts,
                            op["weights"]["grad"], 9 * Ci * Co, (L + 2047) // 2048)
                    if index:
                        self.gemm(1, current, op["weights"]["value"], self.dummy, other,
                                  L, Ci, 9 * Co, H, W, Ci, Co)
                    C = Co
                else:
                    I, O, L, C = op["I"], op["O"], b, op["O"]
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(5, x, current, self.dummy, target, I, O, b, I=I, O=O)
                    if self.backend == "explicit":
                        k.merge_partials[grid(I * O), 256, self.stream](self.parts, op["weights"]["grad"], I * O, 1)
                    self.gemm(4, current, op["weights"]["value"], self.dummy, other, b, I, O, I=I, O=O)
                self.reduce(current, self.dummy, {}, L, C, 3, op["bias"]["grad"], self.dummy)
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, current, op, L, C, 2, op["beta"]["grad"], op["gamma"]["grad"])
                k.bn_backward[grid(E), 256, self.stream](x, current, other, op["gamma"]["value"],
                    op["mean"], op["inv"], op["beta"]["grad"], op["gamma"]["grad"], L, C)
            elif kind == "pool":
                E_in = b * op["H"] * op["W"] * op["C"]
                k.pool_backward[grid(E_in), 256, self.stream](current, op["arg"], other,
                    b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](current, op["mask"], other,
                    np.float32(1 / (1 - op["p"])), E)
            if audit_hook is not None:
                self.stream.synchronize()
                audit_hook(index, op, current, other)
            current, other = other, current

    def update(self, b):
        t = np.float32(self.steps + 1)
        alpha = np.float32(.001) * np.sqrt(np.float32(1) - np.float32(.999)**t) / (
            np.float32(1) - np.float32(.9)**t)
        for p in self.params:
            k.adam[grid(p["value"].size), 256, self.stream](p["value"], p["grad"],
                p["m"], p["v"], np.float32(alpha), p["value"].size, self.error)
        k.metrics[1, 256, self.stream](self.loss, self.correct, b, self.total)
        k.advance_step[1, 1, self.stream](self.counter)
        self.steps += 1

    def train_step(self, x, labels, epoch=0, batch_index=0):
        self.upload(x, labels, epoch, batch_index)
        with self.timed("forward_loss"):
            self.forward(len(x))
        with self.timed("backward"):
            self.backward(len(x))
        with self.timed("adam_metricas"):
            self.update(len(x))
        with self.timed("d2h_control"):
            self.error.copy_to_host(self.host_error, stream=self.stream)
        times = self.drain_events()
        if self.host_error[0]:
            raise FloatingPointError(f"CUDA sticky error {self.host_error[0]}, step {self.steps}")
        return times

    def reset_metrics(self):
        self.total.copy_to_device(np.zeros(3, np.float32), stream=self.stream)

    def export(self, optimizer=False):
        """D2H model state, then install CPU mirror outside the transfer interval."""
        host = []
        with self.timed("d2h_modelo"):
            for p in self.params:
                host.append((p["variable"], p["value"].copy_to_host(stream=self.stream).reshape(p["initial"].shape)))
            for op in self.ops:
                if op["kind"] == "bn":
                    host.append((op["layer"].moving_mean, op["moving_mean"].copy_to_host(stream=self.stream)))
                    host.append((op["layer"].moving_variance, op["moving_var"].copy_to_host(stream=self.stream)))
            self.total.copy_to_host(self.host_total, stream=self.stream)
        times = self.drain_events()
        for variable, value in host:
            variable.assign(value)
        if optimizer:
            with self.timed("d2h_optimizador"):
                states = [(p["m"].copy_to_host(stream=self.stream), p["v"].copy_to_host(stream=self.stream)) for p in self.params]
                count = self.counter.copy_to_host(stream=self.stream)
            extra = self.drain_events()
            times.update(extra)
            if int(count[0]) != self.steps:
                raise ValueError("Device update counter differs")
            self.reference.optimizer.iterations.assign(int(count[0]))
            for i, (moment, velocity) in enumerate(states):
                self.reference.optimizer._momentums[i].assign(moment.reshape(self.params[i]["initial"].shape))
                self.reference.optimizer._velocities[i].assign(velocity.reshape(self.params[i]["initial"].shape))
        return times, self.host_total.copy()


In [ ]:
def locations(experiment):
    if not experiment.startswith('EXP-') or not experiment[4:].isdigit():
        raise ValueError('Invalid experiment identity')
    return (ROOT / 'configuraciones' / (experiment + '.json'), ROOT / 'resultados/paralelo' / experiment)

def load_manifest(config):
    path = ROOT / 'resultados/secuencial/EXP-002/manifest_splits.csv'
    if sha(path) != config['datos']['manifest_splits_sha256']:
        raise ValueError('Split manifest identity changed')
    with path.open(encoding='utf8', newline='') as f:
        rows = list(csv.DictReader(f))
    result = {s: sorted([r for r in rows if r['split'] == s], key=lambda r: int(r['orden'])) for s in ('train', 'val', 'test')}
    for s, n in (('train', 12180), ('val', 2610), ('test', 2610)):
        if len(result[s]) != n:
            raise ValueError('Data count changed')
    return result

def load_images(rows, root, workers=6):
    """Same Pillow transformation as original; fixed order, bounded independent I/O."""
    import numpy as np
    from PIL import Image
    X = np.zeros((len(rows), 64, 64, 3), np.float32)
    y = np.zeros(len(rows), np.int64)

    def read(row):
        with Image.open(root / row['ruta_relativa']) as image:
            value = np.asarray(image.convert('RGB').resize((64, 64)), dtype=np.float32) / 255.0
        return (value, int(row['label_idx']))
    if workers == 1:
        for i, row in enumerate(rows):
            X[i], y[i] = read(row)
    else:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            pending = deque()
            iterator = iter(rows)
            for _ in range(min(12, len(rows))):
                pending.append(pool.submit(read, next(iterator)))
            i = 0
            while pending:
                X[i], y[i] = pending.popleft().result()
                i += 1
                row = next(iterator, None)
                if row is not None:
                    pending.append(pool.submit(read, row))
    return (X, y)

## Configuración efectiva del dispositivo y verificación de entradas

In [ ]:
conf, folder = locations(experiment)

config = json.loads(conf.read_text(encoding='utf8'))

for relative, digest in config['codigo']['fuentes'].items():
    if sha(ROOT / relative) != digest:
        raise ValueError(f'Frozen source changed: {relative}')

for name, value in {'OMP_NUM_THREADS': '6', 'MKL_NUM_THREADS': '6', 'OPENBLAS_NUM_THREADS': '6', 'TF_NUM_INTRAOP_THREADS': '6', 'TF_NUM_INTEROP_THREADS': '1', 'TF_DETERMINISTIC_OPS': '1', 'TF_ENABLE_ONEDNN_OPTS': '1', 'NUMBA_CUDA_LOW_OCCUPANCY_WARNINGS': '0'}.items():
    os.environ[name] = value

import numpy as np

import psutil

from threadpoolctl import threadpool_limits

from asl_cuda.model import Network, cpu_reference, INITIAL, INITIAL_SHA, compiled_resources

from asl_cuda.runtime import cuda

if psutil.virtual_memory().available < 4096 * 2 ** 20:
    raise MemoryError('At least 4GiB available host RAM required')

rows = []

epoch_rows = []

config_sha = sha(conf)

def state(phase, **other):
    write_json(directory / 'estado.json', dict(utc=utc(), fase=phase, **other))
    print(utc(), phase, other, flush=True)

def add(phase, seconds, kind='pared', start='', end='', **load):
    row = dict.fromkeys(FIELDS, '')
    row.update(experimento_id=experiment, run_id=directory.name, dispositivo='gpu', repeticion=repetition, fase=phase, tipo_medicion=kind, duracion_s=float(seconds), inicio_utc=start, fin_utc=end, incluido_resumen='true', estado='completada', configuracion_sha256=config_sha, datos_sha256=config['datos']['manifest_splits_sha256'], evidencia=(directory / 'tiempos.csv').relative_to(ROOT).as_posix())
    row.update(load)
    rows.append(row)
    write_csv(directory / 'tiempos.csv', rows, FIELDS)

state('verificacion_fuentes_y_datos')

manifest = load_manifest(config)

data_root = Path(config['datos']['ruta'])

for split in manifest.values():
    for item in split:
        if sha(data_root / item['ruta_relativa']) != item['sha256']:
            raise ValueError('Dataset file changed: ' + item['ruta_relativa'])

state('inicializacion_y_warmup')

init = time.perf_counter()

tf, reference = cpu_reference()

net = Network(reference, backend='cublas')

write_json(directory / 'hardware.json', dict(device=net.info, cpu=config['cpu'], tensorflow=tf.__version__, tf_build=tf.sysconfig.get_build_info(), tf_visible=[str(d) for d in tf.config.get_visible_devices()], cpu_tensor_device=tf.constant([1.0]).device, cublas_version=net.blas.version, cublas_dll=str(net.blas.path), cublas_sha256=sha(net.blas.path), buffers_bytes=net.bytes_allocated, initial_checkpoint_sha256=sha(INITIAL)))

if sha(INITIAL) != INITIAL_SHA or tf.config.get_visible_devices('GPU'):
    raise ValueError('CPU mirror or initial state invalid')

wx, wy = load_images(manifest['train'][:148], data_root, 1)

wv, wvy = load_images(manifest['val'][:128], data_root, 1)

wt, wty = load_images(manifest['test'][:32], data_root, 1)

In [ ]:
display(Markdown("### Hardware reconocido durante esta ejecución"))
print((directory / "hardware.json").read_text(encoding="utf8"))
print("No es una consulta histórica: el registro acaba de generarse en esta ejecución.")


In [ ]:
threadpool_guard = threadpool_limits(limits=6)
threadpool_guard.__enter__()
print("Límite de bibliotecas matemáticas: 6 hilos")


## Warmup y restauración completa

In [ ]:
net.train_step(wx[:128], wy[:128], 0, 0)

net.train_step(wx[128:], wy[128:], 0, 1)

reference.evaluate(wv, wvy, batch_size=128, verbose=0)

reference.evaluate(wt, wty, batch_size=32, verbose=0)

net.restore()

for p in net.params:
    if not np.array_equal(p['value'].copy_to_host(), p['initial'].ravel()):
        raise ValueError('Warmup parameter restore failed')
    if np.any(p['m'].copy_to_host()) or np.any(p['v'].copy_to_host()):
        raise ValueError('Warmup optimizer restore failed')

if net.counter.copy_to_host()[0] != 0:
    raise ValueError('Warmup counter restore failed')

for op in net.ops:
    if op['kind'] == 'bn':
        if not np.array_equal(op['moving_mean'].copy_to_host(), op['initial_mean']) or not np.array_equal(op['moving_var'].copy_to_host(), op['initial_var']):
            raise ValueError('Warmup BN restore failed')

write_json(directory / 'warmup.json', dict(duracion_s=time.perf_counter() - init, train=148, val=128, test=32, restauracion_verificada=True, metricas_warmup_excluidas=True))

## FASE CPU: lectura y preparación de imágenes

In [ ]:
state('preparacion_cpu', workers=6)

pipeline_start_utc, pipeline_start = (utc(), time.perf_counter_ns())

prep_start = time.perf_counter_ns()

inputs = {s: load_images(manifest[s], data_root, 6) for s in ('train', 'val', 'test')}

prep_end = time.perf_counter_ns()

add('preparacion_cpu', (prep_end - prep_start) / 1000000000.0, start=pipeline_start_utc, end=utc(), muestras=17400, motivo='CPU6workers; orden congelado; no atribuir mejora de esta fase a CUDA')

x, y = inputs['train']

vx, vy = inputs['val']

tx, ty = inputs['test']

## Verificación de tensores frente a CPU

In [ ]:
audit_start = time.perf_counter_ns()

expected = json.loads((ROOT / 'resultados/secuencial/EXP-002/RUN-20261117T171942Z-cpu-01/tensores.json').read_text(encoding='utf8'))

tensor_hash = {}

for split, (images, labels) in inputs.items():
    fingerprint = hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    label_hash = hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    if fingerprint != expected[split]['images_sha256'] or label_hash != expected[split]['labels_sha256']:
        raise ValueError('CPU-prepared tensors differ from historical reference')
    tensor_hash[split] = dict(images_sha256=fingerprint, labels_sha256=label_hash, shape=list(images.shape), dtype=str(images.dtype))

write_json(directory / 'tensores.json', tensor_hash)

audit_seconds = (time.perf_counter_ns() - audit_start) / 1000000000.0

add('auditoria_tensores', audit_seconds, motivo='Anidado en pipeline; fuera de entrenamiento y suma comparable')

In [ ]:
for etiqueta, (imagenes, clases_y) in inputs.items():
    print(etiqueta, "X", imagenes.shape, imagenes.dtype, "y", clases_y.shape, clases_y.dtype,
          "rango", float(imagenes.min()), float(imagenes.max()))
fig, axes = plt.subplots(2,6,figsize=(12,4))
for i,ax in enumerate(axes.flat):
    ax.imshow(x[i]); ax.set_title(str(y[i])); ax.axis("off")
fig.suptitle("Muestras reales tras preparación CPU")
from registro_notebook import mostrar_figura
mostrar_figura(fig)


## FASE GPU: entrenamiento CNN-B, 15 épocas

In [ ]:
state('entrenamiento_CNN-B', epoca=1, pasos=0)

train_start_utc, train_start = (utc(), time.perf_counter_ns())

event_totals = {}

validations = 0.0

export_wall = 0.0

with net.timed('h2d_inicial'):
    net.restore()

for phase, seconds in net.drain_events().items():
    event_totals[phase] = event_totals.get(phase, 0.0) + seconds

rng = np.random.Generator(np.random.PCG64(43))

shuffle_hashes = []

for epoch in range(15):
    epoch_start = time.perf_counter()
    net.reset_metrics()
    order = rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j, begin in enumerate(range(0, len(x), 128)):
        indices = order[begin:begin + 128]
        timings = net.train_step(x[indices], y[indices], epoch, j)
        for phase, seconds in timings.items():
            event_totals[phase] = event_totals.get(phase, 0.0) + seconds
        if j % 12 == 0:
            state('entrenamiento_CNN-B', epoca=epoch + 1, pasos=net.steps)
    export_start = time.perf_counter()
    times, total = net.export()
    export_wall += time.perf_counter() - export_start
    for phase, seconds in times.items():
        event_totals[phase] = event_totals.get(phase, 0.0) + seconds
    if int(total[2]) != 12180:
        raise ValueError('Epoch sample counter differs')
    state('validacion_cpu', epoca=epoch + 1, pasos=net.steps)
    val_start = time.perf_counter()
    val = reference.evaluate(vx, vy, batch_size=128, verbose=0, return_dict=True)
    val_s = time.perf_counter() - val_start
    validations += val_s
    epoch_rows.append(dict(epoca=epoch + 1, pasos_acumulados=net.steps, train_loss=float(total[0] / total[2]), train_accuracy=float(total[1] / total[2]), val_loss=float(val['loss']), val_accuracy=float(val['accuracy']), pared_epoca_s=time.perf_counter() - epoch_start, validacion_cpu_s=val_s))
    write_csv(directory / 'epocas.csv', epoch_rows)
    print('EPOCH', epoch + 1, epoch_rows[-1], flush=True)

times, _ = net.export(optimizer=True)

for phase, seconds in times.items():
    event_totals[phase] = event_totals.get(phase, 0.0) + seconds

train_end, train_end_utc = (time.perf_counter_ns(), utc())

add('entrenamiento_CNN-B', (train_end - train_start) / 1000000000.0, start=train_start_utc, end=train_end_utc, epocas=15, pasos=1440, muestras=182700, batch_size=128, motivo='Incluye H2D,D2H,CPUstaging y validacion CPU')

add('validacion_anidada_CNN-B', validations, epocas=15, muestras=39150, motivo='Subintervalo entrenamiento: no sumar dos veces')

for phase, seconds in event_totals.items():
    add(phase, seconds, kind='evento_device', epocas=15, pasos=1440, motivo='Intervalos stream anidados en entrenamiento; incluyen posibles huecos de envio Host')

## FASE CPU: evaluación final tras D2H

In [ ]:
state('evaluacion_final_cpu', pasos=net.steps)

eval_start_utc, eval_start = (utc(), time.perf_counter_ns())

test = reference.evaluate(tx, ty, batch_size=32, verbose=0, return_dict=True)

eval_seconds = (time.perf_counter_ns() - eval_start) / 1000000000.0

add('evaluacion_CNN-B', eval_seconds, start=eval_start_utc, end=utc(), muestras=2610, motivo='CPU tras D2H del modelo; no atribuir a inferencia CUDA')

pipeline_end, pipeline_end_utc = (time.perf_counter_ns(), utc())

add('pipeline_total_cnn_b', (pipeline_end - pipeline_start) / 1000000000.0, start=pipeline_start_utc, end=pipeline_end_utc, epocas=15, pasos=1440, muestras=17400, batch_size=128, motivo='Continuo: preparacion+auditoria explicita+entrenamiento+evaluacion; no baseline total adaptativo CPU')

if net.steps != 1440 or int(reference.optimizer.iterations.numpy()) != 1440:
    raise ValueError('Incorrect training step count')

metrics = dict(test={k: float(v) for k, v in test.items()}, epocas=15, pasos=net.steps, val_final=epoch_rows[-1], accuracy_historica_cpu_min=0.9287356, comparabilidad='Mismos datos,modelo,pasos; RNG/ejecucion diferentes, no identidad numerica')

write_json(directory / 'metricas.json', metrics)

write_json(directory / 'transferencias.json', dict(lotes=net.transfers[-1440:], h2d_estado_inicial_bytes=3 * 2245245 * 4 + 960 * 4 + 24, reinicios_metricas_h2d_bytes=15 * 12, modelo_d2h_copias=16, modelo_bytes_por_copia=2246205 * 4, optimizador_final_bytes=2 * 2245245 * 4 + 8, control_d2h_bytes=1440 * 4, aclaracion='Ultima copia extra de pesos por export final, incluida en pared y eventos'))

write_json(directory / 'ordenes.json', dict(seed=43, generador='PCG64', epocas_sha256=shuffle_hashes))

## Persistencia del modelo y configuración de kernels

In [ ]:
state('guardado_fuera_de_timing', pasos=net.steps)

out = ROOT / 'data' / experiment / directory.name

out.mkdir(parents=True, exist_ok=False)

model_path = out / 'CNN-B-CUDA.keras'

reference.save(model_path)

write_json(directory / 'modelo-guardado.json', dict(ruta=model_path.relative_to(ROOT).as_posix(), sha256=sha(model_path), optimizer_iterations=1440, nota='Inferencia Keras CPU; reanudar entrenamiento CUDA requiere claves RNG y estado del adaptador'))

geometries = []

for op in net.ops:
    if op['kind'] == 'conv':
        geometries.append(dict(operacion='im2col', block=[256, 1, 1], grid=[min(4096, (128 * op['H'] * op['W'] * 9 * op['Ci'] + 255) // 256), 1, 1], forma=[128, op['H'], op['W'], op['Ci'], op['Co']]))

write_json(directory / 'lanzamientos.json', dict(propio=geometries, vector_block=256, reduccion_block=[32, 8, 1], reduccion_final_block=256, softmax_block=32, gemm='cuBLAS SGEMM: jerarquia interna gestionada por NVIDIA, no inventada por el equipo', recursos=compiled_resources()))

net.stream.synchronize()

net.blas.close()

state('completada', pasos=1440)

## Resultados completos de esta ejecución

In [ ]:
print((directory / "metricas.json").read_text(encoding="utf8"))
display(pd.read_csv(directory / "tiempos.csv"))
display(pd.read_csv(directory / "epocas.csv"))


In [ ]:
reference.summary()
epocas = pd.read_csv(directory / "epocas.csv")
fig, axes = plt.subplots(1,2,figsize=(11,4))
for name in ("train_accuracy","val_accuracy"): axes[0].plot(epocas.epoca,epocas[name],label=name)
for name in ("train_loss","val_loss"): axes[1].plot(epocas.epoca,epocas[name],label=name)
for ax in axes: ax.set_xlabel("Época");ax.grid(alpha=.15);ax.legend()
axes[0].set_ylabel("Accuracy");axes[1].set_ylabel("Pérdida");fig.tight_layout();mostrar_figura(fig)
for nombre in ("transferencias.json","lanzamientos.json","modelo-guardado.json","ordenes.json"):
    print("\n",nombre);print((directory/nombre).read_text(encoding="utf8"))


## Recursos utilizados, logs y cierre de la ejecución nueva

In [ ]:
monitor_stop.set(); monitor_thread.join(timeout=15)
threadpool_guard.__exit__(None,None,None)
recursos_actuales = pd.read_csv(directory / "recursos.csv")
display(recursos_actuales)
columnas_cpu=[c for c in recursos_actuales if c.startswith("cpu_logico_")]
fig, axes = plt.subplots(2,1,figsize=(12,6))
for col in columnas_cpu: axes[0].plot(recursos_actuales.transcurrido_s,recursos_actuales[col],label=col)
axes[0].set(ylabel="CPU por lógico (%)",title="Uso real registrado durante esta ejecución")
axes[0].legend(ncol=4,fontsize=7)
axes[1].plot(recursos_actuales.transcurrido_s,recursos_actuales.gpu_util_pct,label="Actividad GPU (%)")
axes[1].plot(recursos_actuales.transcurrido_s,recursos_actuales.gpu_mem_used_mib/4096*100,label="VRAM ocupada (%)")
axes[1].set(xlabel="Tiempo de proceso (s)",ylabel="Porcentaje");axes[1].legend();fig.tight_layout();mostrar_figura(fig)
for captura in sorted(directory.glob("captura-*.png")):
    display(Image(filename=str(captura)))
    print(captura.name, (captura.with_suffix(".json")).read_text(encoding="utf8"))
write_json(directory / "salida.json",dict(codigo=0,fin_utc=utc(),tipo="ejecucion_directa_notebook"))
print("EJECUCIÓN NUEVA COMPLETA:", directory.name)


## Cinco mediciones históricas del informe — registros íntegros

Esta sección es histórica y está separada de la corrida nueva anterior. Se muestran stdout/stderr completos,épocas,métricas y tiempos de cada RUN. No se vuelven a entrenar ni se mezclan con el promedio original.

### Registro histórico RUN-20261117T231628Z-gpu-01

In [ ]:
historico = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T231628Z-gpu-01"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T232412Z-gpu-02

In [ ]:
historico = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T232412Z-gpu-02"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T233146Z-gpu-03

In [ ]:
historico = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T233146Z-gpu-03"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T233923Z-gpu-04

In [ ]:
historico = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T233923Z-gpu-04"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


### Registro histórico RUN-20261117T234658Z-gpu-05

In [ ]:
historico = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T234658Z-gpu-05"
print("HISTÓRICO:",historico.name)
print((historico/"stdout.log").read_text(encoding="utf8"))
print("STDERR ORIGINAL:")
print((historico/"stderr.log").read_text(encoding="utf8"))
display(pd.read_csv(historico/"epocas.csv"))
display(pd.read_csv(historico/"tiempos.csv"))
print((historico/"metricas.json").read_text(encoding="utf8"))


## Comparación histórica del informe (no incluye la corrida nueva)

In [ ]:
display(pd.read_csv(ROOT/"resultados/paralelo/EXP-003/comparacion_cnn_b.csv"))
display(pd.read_csv(ROOT/"resultados/paralelo/EXP-003/speedup.csv"))
print("Solo CNN-B base comparable. Preparación1→6workers es mejoraCPU;referencia histórica,RNG distinto y sin afirmación de saturaciónSM/VRAM.")
